In [ ]:
# # Install dependencies for Unsloth + GPT-OSS
# !pip install --upgrade -qqq uv
# !uv pip install -qqq \
#     "torch>=2.8.0" "triton>=3.4.0" numpy pillow torchvision bitsandbytes "transformers==4.56.2" \
#     "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#     "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#     git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# # Install openai_harmony (Harmony protocol tools)
# !pip install -q openai-harmony jupyter_client pandas datasets


In [ ]:
class CONFIG:
    TRAIN_SIZE = 50_000
    BATCH_SIZE = 1
    EPOCHS = 1 
    LEARNING_RATE = 1e-4
    MAX_SEQ_LENGTH = 8_000
    MODEL_PATH = None 
    KAGGLE=True 
    MASK_THINK = False

cfg = CONFIG()
cfg.MODEL_PATH = "/kaggle/input/models/barnobarno/gpt-oss-120b-bnb-4bit/transformers/unsloth/1" if cfg.KAGGLE else "unsloth/gpt-oss-20b"

In [ ]:
!python --version

In [ ]:
print("STARTING THE AHHHHHHHHHHHHHHHHHHHHHHHHHHHH")

In [ ]:
if cfg.KAGGLE:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-py-3-12/unsloth' 'unsloth'


In [ ]:
try:
    from unsloth import FastLanguageModel
except:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-library/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

    

In [ ]:
local_files_only = True if cfg.KAGGLE else False

In [ ]:
# del model
# del tokenizer

In [ ]:
import torch
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.MODEL_PATH ,
    dtype = dtype, # None for auto detection
    max_seq_length = cfg.MAX_SEQ_LENGTH, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    local_files_only = local_files_only
)

In [ ]:
import pandas as pd
if not cfg.KAGGLE:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    # Set the path to the file you'd like to load
    file_path = "filtered_low_pass_1_or_2.jsonl"

    # Load the latest version
    df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "barnobarno/nemotron-low-reasoning-pass-rate-1-2",
    file_path,
    # Provide any additional arguments like 
    # sql_query or pandas_kwargs. See the 
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
    )
if cfg.KAGGLE:
    df = pd.read_json("/kaggle/input/nemotron-low-reasoning-pass-rate-1-2/filtered_low_pass_1_or_2.jsonl", lines=True)
print("DONE DATA LOADING")


In [ ]:
from datasets import Dataset

train_data = df.copy()
train_data.drop(columns=["uuid","original_expected_answer","license" ,"used_in" ,"user_name" ,"user_url" ,"url"], inplace=True ,axis=1)
#train_data.dropna(inplace=True)
train_data = train_data[train_data["tools"].isna()]
train_data.drop(columns=["tools"], inplace=True ,axis=1)
def remove_none_keys(messages):
    return [{k: v for k, v in entry.items() if v is not None} for entry in messages]
def format_for_gpt_oss(example):
    messages = example['messages']
    new_messages = []
    
    for msg in messages:
        new_msg = msg.copy()
        
        # 1. Rename 'reasoning_content' to 'thinking'
        if 'reasoning_content' in new_msg:
            #pass 
            new_msg['thinking'] = new_msg.pop('reasoning_content')
        
        # 2. Ensure intermediate tool steps don't conflict
        # The template raises an error if you have BOTH 'thinking' and 'content' 
        # inside a tool call message. Your data has content='', which is fine, 
        # but purely safe practice is to ensure it is None or empty.
        if new_msg.get('tool_calls') and new_msg.get('thinking'):
             new_msg['content'] = "" # Ensure this is empty to avoid template error

        new_messages.append(new_msg)
    
    return {'messages': new_messages}

# Apply to your dataset
train_data_formatted = train_data.apply(format_for_gpt_oss, axis=1)
train_data["messages"] = train_data_formatted
train_data["messages"].iloc[0]["messages"]
train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])


# 1. Shuffle first (Optional but recommended)
# This ensures you don't just pick the "first" solution for a problem if there are duplicates.
train_data_shuffled = train_data.copy()  #  .sample(frac=1, random_state=42)

# 2. Get the pool of unique problems
# This drops all duplicates, leaving you with exactly 590k rows (1 per unique answer)
unique_pool = train_data_shuffled.drop_duplicates(subset=["expected_answer"])
print(f"Unique Shape: {unique_pool.shape}")

# 3. Sample your target 100k from that pool
dataset = unique_pool.sample(n=cfg.TRAIN_SIZE, random_state=42)

# Both should be 100,000


#dataset = train_data.iloc[0:cfg.TRAIN_SIZE].copy()

dataset["AA"] = dataset["AA"].apply(remove_none_keys)

dataset["text"] = dataset.apply(lambda row: tokenizer.apply_chat_template(
    row["AA"], 
    tokenize=False, 
    add_generation_prompt=False,
    reasoning_effort="low"
), axis=1)
print(f"Final dataset shape: {dataset.shape}")
print(f"Unique answers: {dataset['expected_answer'].nunique()}")


In [ ]:
dataset["messages"].iloc[20]

In [ ]:
dataset["text"].iloc[20]

In [ ]:
len(train_data[train_data["changed_answer_to_majority"]==False])

In [ ]:
# Fastest way to get the COUNT of unique items
count = train_data["problem"].nunique()
print(f"Unique items: {count}")

In [ ]:
# from datasets import Dataset

# # 1. Pre-processing
# train_data = df.copy()
# train_data.drop(columns=["uuid", "original_expected_answer", "license", "used_in", "user_name", "user_url", "url"], inplace=True, axis=1)

# # Filter out rows with tools (as per your snippet)
# train_data = train_data[train_data["tools"].isna()]
# train_data.drop(columns=["tools"], inplace=True, axis=1)

# def remove_none_keys(messages):
#     return [{k: v for k, v in entry.items() if v is not None} for entry in messages]

# def format_for_gpt_oss(example):
#     messages = example['messages']
    
#     # --- MODIFICATION START ---
#     # Initialize with the System Prompt
#     new_messages = [{
#         "role": "system", 
#         "content": "YOU ARE A MATH EXPERT"
#     }]
#     # --- MODIFICATION END ---
    
#     for msg in messages:
#         # Optional: Skip existing system prompts to strictly enforce your new one
#         if msg.get('role') == 'system':
#             continue

#         new_msg = msg.copy()
        
#         # 1. Rename 'reasoning_content' to 'thinking'
#         if 'reasoning_content' in new_msg:
#             new_msg['thinking'] = new_msg.pop('reasoning_content')
        
#         # 2. Ensure intermediate tool steps don't conflict
#         if new_msg.get('tool_calls') and new_msg.get('thinking'):
#              new_msg['content'] = "" 

#         new_messages.append(new_msg)
    
#     return {'messages': new_messages}

# # Apply to your dataset
# train_data_formatted = train_data.apply(format_for_gpt_oss, axis=1)
# train_data["messages"] = train_data_formatted

# # Extract the list of messages
# train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])

# # Create the final dataset slice
# dataset = train_data.iloc[0:cfg.TRAIN_SIZE].copy()
# dataset["AA"] = dataset["AA"].apply(remove_none_keys)

# # Apply Chat Template
# dataset["text"] = dataset.apply(lambda row: tokenizer.apply_chat_template(
#     row["AA"], 
#     tokenize=False, 
#     add_generation_prompt=True,
#     reasoning_effort="low"
# ), axis=1)

In [ ]:
# Add LoRA adapters with rank 16
hf_dataset = Dataset.from_pandas(dataset)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
import gc
import torch
gc.collect()


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=cfg.BATCH_SIZE,
        gradient_accumulation_steps=2,
        warmup_steps=5,
        #num_train_epochs=1, 
        max_steps=500 ,
        learning_rate=cfg.LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=666,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        group_by_length=True
    ),
)



In [ ]:
# CHANGE THIS: Point to the analysis channel instead of the final channel sometime
if not cfg.MASK_THINK:
    try:
        gpt_oss_kwargs = dict(
            instruction_part = "<|start|>user<|message|>", 
            response_part = "<|start|>assistant<|channel|>analysis<|message|>"
        )
        
        trainer = train_on_responses_only(
            trainer,
            **gpt_oss_kwargs,
        )
        TRAIN=True
    except:
        TRAIN = False
else:
    gpt_oss_kwargs = dict(
    instruction_part="<|start|>user<|message|>", 
    response_part="<|start|>assistant<|channel|>final<|message|>"
)
    trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)
    TRAIN = True



    


print(f"TRAINING READY AS NOT MASKING THE THINKING FROM TRAIN LOSS {cfg.MASK_THINK}")

In [ ]:
# 1. Train the model
if TRAIN:
    trainer_stats = trainer.train()

# 2. Memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# 3. Time stats
# 'train_runtime' is in seconds
train_time_seconds = trainer_stats.metrics.get('train_runtime', 0)
train_time_minutes = round(train_time_seconds / 60, 2)

print(f"Peak reserved memory = {used_memory} GB")
print(f"Total training time  = {train_time_minutes} minutes ({train_time_seconds:.2f} seconds)")

In [ ]:
model.save_pretrained("gpt_oss_20b_nemotronv2_low_FIXED")
tokenizer.save_pretrained("gpt_oss_20b_nemotronv2_low_FIXED")
print("Model saved to 'gpt_oss_20b_nemotronv2_low'")